# Trabalho 2 - CG

### Imports

In [232]:
!pip install pyopengl
!pip install glfw
!pip install pyglm
!pip install numpy
!pip install pillow

/home/henriquedrago/Documents/7° Periodo/CG/Trabalho 2/.venv/bin/pip: line 2: /home/henriquedrago/Documents/7° Periodo/CG/SCC0250-Computacao-Grafica-Trabalho-02/.venv/bin/python: No such file or directory


/home/henriquedrago/Documents/7° Periodo/CG/Trabalho 2/.venv/bin/pip: line 2: /home/henriquedrago/Documents/7° Periodo/CG/SCC0250-Computacao-Grafica-Trabalho-02/.venv/bin/python: No such file or directory
/home/henriquedrago/Documents/7° Periodo/CG/Trabalho 2/.venv/bin/pip: line 2: /home/henriquedrago/Documents/7° Periodo/CG/SCC0250-Computacao-Grafica-Trabalho-02/.venv/bin/python: No such file or directory
/home/henriquedrago/Documents/7° Periodo/CG/Trabalho 2/.venv/bin/pip: line 2: /home/henriquedrago/Documents/7° Periodo/CG/SCC0250-Computacao-Grafica-Trabalho-02/.venv/bin/python: No such file or directory
/home/henriquedrago/Documents/7° Periodo/CG/Trabalho 2/.venv/bin/pip: line 2: /home/henriquedrago/Documents/7° Periodo/CG/SCC0250-Computacao-Grafica-Trabalho-02/.venv/bin/python: No such file or directory


In [233]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
import os
from numpy import random
from PIL import Image
import json

from shader_s import Shader

### Inicializando janela

In [234]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 700
largura = 700

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)



(python:6257): Gtk-WARNING **: 20:32:42.673: gtk_disable_setlocale() must be called before gtk_init()


### Shaders

In [235]:
mainShader = Shader("vertex_shader.vs", "fragment_shader.fs")
skyboxShader = Shader("skybox.vs", "skybox.fs")
# ourShader.use()

program = mainShader.getProgram()
skyboxProgram = skyboxShader.getProgram()

### Carregando Modelos

In [236]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []


def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r"): ## para cada linha do arquivo .obj
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        ### recuperando vertices
        if values[0] == 'v':
            vertices.append(values[1:4])

        ### recuperando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model

def load_texture_from_file(img_textura):
    texture_id = glGenTextures(1) # Pede um id a opengl
    
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    img = Image.open(img_textura)
    img = img.convert('RGBA') # Converte pra RGBA
    
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGBA", 0, -1)
    
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, img_width, img_height, 0, GL_RGBA, GL_UNSIGNED_BYTE, image_data)
    
    return texture_id # Retorna o ID


'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result
    
global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[2] not in faces_visited:
            faces_visited.append(face[2])
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
        for texture_id in circular_sliding_window_of_three(face[1]):
            textures_coord_list.append(modelo['texture'][texture_id - 1])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))
    
    tid = load_texture_from_file(texturesList[0])
    
    return verticeInicial, verticeFinal - verticeInicial, tid

In [237]:
def load_mtl(mtl_path):
    """
    Lê um arquivo .mtl e retorna um dicionário de materiais.
    Formato retornado: {nome_material: {'map_Kd': caminho_textura_ou_None, 'Kd': [r, g, b]}}
    """
    materials = {}
    current = None
    mtl_dir = os.path.dirname(os.path.abspath(mtl_path))

    for line in open(mtl_path, 'r', encoding='utf-8', errors='ignore'):
        if line.startswith('#'):
            continue
        values = line.split()
        if not values:
            continue
        if values[0] == 'newmtl':
            current = values[1]
            materials[current] = {'map_Kd': None, 'Kd': [1.0, 1.0, 1.0]}
        elif values[0] == 'Kd' and current:
            materials[current]['Kd'] = [float(v) for v in values[1:4]]
        elif values[0] == 'map_Kd' and current:
            # suporta caminhos com espaços e separadores diferentes de SO
            tex_rel = ' '.join(values[1:]).strip().replace('\\', os.sep).replace('/', os.sep)
            materials[current]['map_Kd'] = os.path.join(mtl_dir, tex_rel)

    return materials


def load_obj_with_mtl(obj_path, texture_overrides=None):
    """
    Carrega um modelo .obj com múltiplos materiais definidos em um arquivo .mtl.

    Parâmetros:
        obj_path          : caminho para o arquivo .obj
        texture_overrides : dict {nome_material: caminho_textura}
                            Use quando o .mtl não possui entradas 'map_Kd', mas
                            as texturas existem em pastas separadas.
                            Exemplo para o modelo da casa:
                            {
                                'concrete.002': 'objetos/casa/textures/concrete/concrete_bc.jpg',
                                'redwood.002' : 'objetos/casa/textures/tiled_redwood/T_tiled_redwood_bc.png',
                                'roof.002'    : 'objetos/casa/textures/roof/roof_bc.png',
                            }

    Retorna:
        Lista de grupos de renderização, onde cada grupo é um dict:
        {'vertice_inicial': int, 'num_vertices': int, 'texture_id': int}
        Um grupo é criado por material encontrado no .obj.
    """
    global vertices_list, textures_coord_list

    obj_dir = os.path.dirname(os.path.abspath(obj_path))

    # ── 1. Parse do arquivo OBJ ──────────────────────────────────────────────
    raw_vertices   = []
    raw_tex_coords = []
    faces_por_material = {}   # {nome_material: [(face_verts, face_uvs), ...]}
    material_atual = '__default__'
    mtl_files = []

    for line in open(obj_path, 'r', encoding='utf-8', errors='ignore'):
        if line.startswith('#'):
            continue
        values = line.split()
        if not values:
            continue

        if values[0] == 'v':
            raw_vertices.append(values[1:4])
        elif values[0] == 'vt':
            raw_tex_coords.append(values[1:3])
        elif values[0] == 'mtllib':
            mtl_files.append(os.path.join(obj_dir, ' '.join(values[1:])))
        elif values[0] in ('usemtl', 'usemat'):
            material_atual = values[1]
            if material_atual not in faces_por_material:
                faces_por_material[material_atual] = []
        elif values[0] == 'f':
            face_v, face_uv = [], []
            for token in values[1:]:
                parts = token.split('/')
                face_v.append(int(parts[0]))
                face_uv.append(int(parts[1]) if len(parts) >= 2 and parts[1] else 0)
            if material_atual not in faces_por_material:
                faces_por_material[material_atual] = []
            faces_por_material[material_atual].append((face_v, face_uv))

    # ── 2. Parse dos arquivos .mtl referenciados ─────────────────────────────
    materials = {}
    for mtl_path_ref in mtl_files:
        if os.path.exists(mtl_path_ref):
            materials.update(load_mtl(mtl_path_ref))
        else:
            print(f"[AVISO] MTL não encontrado: {mtl_path_ref}")

    # Sobrescrever/completar com caminhos manuais (necessário quando .mtl não tem map_Kd)
    if texture_overrides:
        for mat_name, tex_path in texture_overrides.items():
            if mat_name not in materials:
                materials[mat_name] = {'map_Kd': None, 'Kd': [1.0, 1.0, 1.0]}
            materials[mat_name]['map_Kd'] = tex_path

    # ── 3. Inserir vértices por grupo e carregar textura de cada material ─────
    grupos = []

    for mat_name, faces in faces_por_material.items():
        if not faces:
            continue

        vi_inicio = len(vertices_list)

        for face_v, face_uv in faces:
            for vid in circular_sliding_window_of_three(face_v):
                vertices_list.append(raw_vertices[vid - 1])
            for uid in circular_sliding_window_of_three(face_uv):
                if uid > 0:
                    textures_coord_list.append(raw_tex_coords[uid - 1])
                else:
                    textures_coord_list.append(['0.0', '0.0'])

        nv = len(vertices_list) - vi_inicio

        tex_path = (materials.get(mat_name) or {}).get('map_Kd')

        if tex_path and os.path.exists(tex_path):
            # Recebe o ID seguro diretamente da função atualizada:
            tid = load_texture_from_file(tex_path)
            print(f"  Material '{mat_name}': {nv} vértices → textura id={tid} ({os.path.basename(tex_path)})")
        else:
            # Usa ID 0 (neutro) em vez de reutilizar a textura do objeto anterior:
            tid = 0
            print(f"  [AVISO] Material '{mat_name}': sem textura, usando fallback (id=0)")

        grupos.append({'vertice_inicial': vi_inicio, 'num_vertices': nv, 'texture_id': tid})

    print(f"Modelo '{os.path.basename(obj_path)}' carregado: {len(grupos)} grupo(s) de material")
    return grupos

In [238]:
def load_cubemap(faces):
    # Carrega 6 imagens e as mapeia para as faces de um OpenGL Cubemap.
    # A ordem da lista 'faces' é: Right, Left, Top, Bottom, Front, Back.
    
    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_CUBE_MAP, texture_id)

    for i in range(len(faces)):
        try:
            img = Image.open(faces[i])

            img = img.convert('RGBA') # Converte pra RGBA

            img_width = img.size[0]
            img_height = img.size[1]
            image_data = img.tobytes("raw", "RGBA", 0, 1)
            
            # POSITIVE_X é o primeiro target. Somar 'i' nos leva aos próximos targets automaticamente.
            glTexImage2D(GL_TEXTURE_CUBE_MAP_POSITIVE_X + i, 
                         0, GL_RGBA, img_width, img_height, 0, GL_RGBA, GL_UNSIGNED_BYTE, image_data)
        except Exception as e:
            print(f"Erro ao carregar textura da face {faces[i]}: {e}")

    # Configurações de filtragem
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    # clamp to edge impede que apareçam linhas pretas nas quinas do cubo.
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_R, GL_CLAMP_TO_EDGE)

    return texture_id

### Controlador de cena

In [239]:
# Carrega os dados da cena a partir de arquivos .json
def load_and_merge_scenes(filenames):
    merged_scene = []
    
    for filename in filenames:
        try:
            with open(filename, "r", encoding="utf-8") as f:
                scene_data = json.load(f)

                # Write the source file name in the objects
                for obj in scene_data:
                    obj["scene_file"] = filename

                # Add to the merged scene
                merged_scene.extend(scene_data)

        except FileNotFoundError:
            print(f"Error: File {filename} not found.")
        except json.JSONDecodeError:
            print(f"Error: Could not decode JSON in {filename}.")
            
    return merged_scene

# Salva os dados da cena em seus respectivos arquivos .json
def save_scene(scene):
    # Coleta todos os nomes de arquivo únicos, usando 'scene.json' como fallback
    filenames = {obj.get('scene_file', 'scene.json') for obj in scene}

    for fn in filenames:
        clean_scene = [
            obj for obj in scene
            if obj.get('scene_file', 'scene.json') == fn # joga no scene.json por padrão se não existir arquivo no objeto 
        ]

        try:
            with open(fn, "w", encoding="utf-8") as f:
                json.dump(clean_scene, f, ensure_ascii=False, indent=4)

        except FileNotFoundError:
            print(f"Error: File {fn} not found.")


# Adiciona um objeto à cena
def add_to_scene(scene, obj_id, name, obj_file, texture_file=None, mtl_texture_overrides=None):
    """
    Adiciona um objeto à cena.

    Modelo com textura única:
        add_to_scene(scene, id, "nome", "modelo.obj", texture_file="tex.jpg")

    Modelo com múltiplos materiais (MTL sem map_Kd):
        add_to_scene(scene, id, "nome", "modelo.obj", mtl_texture_overrides={
            "material_1": "caminho/textura1.jpg",
            "material_2": "caminho/textura2.png",
        })
    """
    newObj = {
        "name": name,
        "obj_file": obj_file,
        "texture_file": texture_file,
        "mtl_texture_overrides": mtl_texture_overrides,
        "angulo_obj": 0.0,
        "translacao": [0.0, 0.0, -20.0],
        "escala": [1.0, 1.0, 1.0],
        "rotacao": [0.0, 0.0, 0.0],
        "id_textura": obj_id,
        "obj_id": obj_id,
        "visible": True,
        "polygon": False,
        "scene_id": len(scene),
    }
    scene.append(newObj)
    return scene

In [240]:
# Carrega a cena
scene_files = ["scene.json", "road.json", "chao.json"]
scene = load_and_merge_scenes(scene_files)

In [241]:
# add_to_scene(
#     scene,
#     obj_id=len(scene),      
#     name="Parede_Concreto", 
#     obj_file="objetos/wood-wall/WOOD_WALL.obj", 
#     mtl_texture_overrides={
#         "Material": "objetos/wood-wall/Plaster002_4K-JPG_Color.jpg",
#         "Material.001": "objetos/wood-wall/Plaster002_4K-JPG_Color.jpg"
#     }
# )
# save_scene(scene)


In [242]:


faces_skybox = [
    "objetos/skybox/clouds1_east.bmp",
    "objetos/skybox/clouds1_west.bmp",
    "objetos/skybox/clouds1_up.bmp",
    "objetos/skybox/clouds1_down.bmp",
    "objetos/skybox/clouds1_north.bmp",
    "objetos/skybox/clouds1_south.bmp"
]

# Carrega o cubemap e salva o ID na variável
cubemap_texture_id = load_cubemap(faces_skybox)

### Carregamento dos Objetos da Cena

In [243]:
def carregar_objs():
    # Dicionario pra memorizar os objetos já carregados anteriormente
    cache_modelos = {} 

    triple_list = []

    for object in scene:
        # Chave do dicionário = arquivo_obj + mtl_overrides + arquivo_textura
        overrides_str = str(object.get('mtl_texture_overrides'))
        tex_str = str(object.get('texture_file'))
        cache_key = f"{object['obj_file']}_{overrides_str}_{tex_str}"

        # Verifica se esse modelo já foi lido antes
        if cache_key in cache_modelos:
            grupos = cache_modelos[cache_key]
            
        else:
            # Se não estiver, lê o arquivo
            if object.get('mtl_texture_overrides') is not None:
                print(f"Lendo (MLT_overrides): {object['obj_file']}")
                grupos = load_obj_with_mtl(object['obj_file'], object['mtl_texture_overrides'])
            else:
                print(f"Lendo (no_override): {object['obj_file']}")
                vi, nv, tid = load_obj_and_texture(object['obj_file'], [object['texture_file']])
                grupos = [{'vertice_inicial': vi, 'num_vertices': nv, 'texture_id': tid}]
                
            # Salva na cache
            cache_modelos[cache_key] = grupos

        triple_list.append(grupos)
    
    return triple_list

verticeInicial_quantosVertices_list = carregar_objs()


Lendo (MLT_overrides): objetos/bus_stop/bus_stop.obj
  Material 'blinn1SG': 1149120 vértices → textura id=2 (BusStopEnclosure.jpg)
  Material 'blinn2SG': 16632 vértices → textura id=3 (BusStopEnclosure.jpg)
Modelo 'bus_stop.obj' carregado: 2 grupo(s) de material
Lendo (MLT_overrides): objetos/onibus/bus_byjoao3DModels.obj
  Material 'base2': 211944 vértices → textura id=4 (base2_BaseColor.png)
  Material 'bus': 148782 vértices → textura id=5 (bus_BaseColor.png)
  Material 'glass': 3318 vértices → textura id=6 (glass_op.png)
Modelo 'bus_byjoao3DModels.obj' carregado: 3 grupo(s) de material
Lendo (MLT_overrides): objetos/oven/10122_Microwave_Oven_v1_L3.obj
  Material '10122_Microwave_Oven_v1_SG': 90864 vértices → textura id=7 (10122_Microwave_Oven_v1_Diffuse_SG.jpg)
Modelo '10122_Microwave_Oven_v1_L3.obj' carregado: 1 grupo(s) de material
Lendo (MLT_overrides): objetos/gnome/garden_gnome_4k.obj
  Material 'garden_gnome_01': 127854 vértices → textura id=8 (garden_gnome_diff_4k.jpg)
Modelo

### Desenha Objetos

In [244]:
def desenha_obj(angle, rot_coords, transl_coords, escal_coords, grupos):
    """
    Desenha um objeto 3D aplicando a matriz model calculada a partir das transformações.

    grupos: lista de grupos de renderização, cada um com:
            {'vertice_inicial': int, 'num_vertices': int, 'texture_id': int}
    Modelos com uma única textura têm lista de 1 elemento.
    Modelos com múltiplos materiais têm um elemento por material.
    """
    mat_model = model(angle, *rot_coords, *transl_coords, *escal_coords)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    for grupo in grupos:
        glBindTexture(GL_TEXTURE_2D, grupo['texture_id'])
        glDrawArrays(GL_TRIANGLES, grupo['vertice_inicial'], grupo['num_vertices'])

### Envia Dados para a CPU

In [245]:
buffer_VBO = glGenBuffers(2) # Cria buffers

#### Enviando coordenadas de vértices para a GPU

In [246]:
if(len(scene) != 0):
    vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
    vertices['position'] = vertices_list


    # Upload data
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
    glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
    stride = vertices.strides[0]
    offset = ctypes.c_void_p(0)
    loc_vertices = glGetAttribLocation(program, "position")
    glEnableVertexAttribArray(loc_vertices)
    glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

#### Enviando coordenadas de textura para a GPU

In [247]:
if(len(scene) != 0):
    textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
    textures['position'] = textures_coord_list


    # Upload data
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
    glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
    stride = textures.strides[0]
    offset = ctypes.c_void_p(0)
    loc_texture_coord = glGetAttribLocation(program, "texture_coord")

    glEnableVertexAttribArray(loc_texture_coord)
    glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


### SkyBox

In [248]:
skybox_vertices = np.array([
    # Positions (X, Y, Z)
    -1.0,  1.0, -1.0,
    -1.0, -1.0, -1.0,
     1.0, -1.0, -1.0,
     1.0, -1.0, -1.0,
     1.0,  1.0, -1.0,
    -1.0,  1.0, -1.0,

    -1.0, -1.0,  1.0,
    -1.0, -1.0, -1.0,
    -1.0,  1.0, -1.0,
    -1.0,  1.0, -1.0,
    -1.0,  1.0,  1.0,
    -1.0, -1.0,  1.0,

     1.0, -1.0, -1.0,
     1.0, -1.0,  1.0,
     1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,
     1.0,  1.0, -1.0,
     1.0, -1.0, -1.0,

    -1.0, -1.0,  1.0,
    -1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,
     1.0, -1.0,  1.0,
    -1.0, -1.0,  1.0,

    -1.0,  1.0, -1.0,
     1.0,  1.0, -1.0,
     1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,
    -1.0,  1.0,  1.0,
    -1.0,  1.0, -1.0,

    -1.0, -1.0, -1.0,
    -1.0, -1.0,  1.0,
     1.0, -1.0, -1.0,
     1.0, -1.0, -1.0,
    -1.0, -1.0,  1.0,
     1.0, -1.0,  1.0
], dtype=np.float32)

# Buffer para a SkyBox
skybox_VBO = glGenBuffers(1)

# Upload dos vértices do skybox para a GPU
glBindBuffer(GL_ARRAY_BUFFER, skybox_VBO)
glBufferData(GL_ARRAY_BUFFER, skybox_vertices.nbytes, skybox_vertices, GL_STATIC_DRAW)

### Eventos

In [249]:
# Limites da Câmera
CAM_MAX_X = 20.0
CAM_MIN_X = -20.0
CAM_MAX_Y = 30.0
CAM_MIN_Y = -1.7
CAM_MAX_Z = 0.0
CAM_MIN_Z = -60.0

eixo = 1 # Eixo selecionado para rotação/escala
selected_obj = 0 # Objeto selecionado para a manipulação
uniform_scale = True # Toggle para escala uniforme
PolygonMode = False # Toggle para modo polígono
restrictMode = True # Toggle para modo restrito 

# camera
cameraPos   = glm.vec3(0.0, 0.0, -50.0) # Posição Inicial da Câmera
cameraFront = glm.vec3(0.0, 0.0, 1.0) # Frente Inicial da Câmera
cameraUp    = glm.vec3(0.0, 1.0, 0.0) # Onde é cima

fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0

firstMouse = True
yaw = 90.0 
pitch = 0.0
lastX =  largura/2
lastY =  altura/2

def key_event(window,key,scancode,action,mods):
    global eixo, selected_obj, scene, uniform_scale, PolygonMode, groups, restrictMode
    global cameraPos, cameraFront, cameraUp
    global onibus_acc, scene_files, spinner_acc, spinner_spd, spinner_max_speed, onibus_max_acc, alarm_on, onibus_spd

    # Manipulações Gerais
    # Fechar janela
    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)
    # Toggle Polygon Mode
    if key == glfw.KEY_P and action == glfw.PRESS: 
        PolygonMode = not PolygonMode
    # Reset
    if mods and glfw.MOD_CONTROL:
        if key == glfw.KEY_R and action == glfw.PRESS: 
            # Reseta a cena para o conteúdo dos arquivos .json
            scene = load_and_merge_scenes(scene_files)
            # Reseta parâmetros dos objetos
            spinner_spd = 0
            spinner_acc = 0
            alarm_on = False
            onibus_acc = 0
            onibus_spd = 0

    # Restrict Mode
    if mods and (glfw.MOD_CONTROL & mods) and (glfw.MOD_SHIFT & mods) and (glfw.MOD_ALT & mods) and (glfw.MOD_SUPER & mods):
        if key == glfw.KEY_K and action == glfw.PRESS: 
            restrictMode = not restrictMode
    
    # Movimentação da câmera
    cameraSpeed = 20 * deltaTime
    if key == glfw.KEY_H and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    
    if key == glfw.KEY_N and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    
    if key == glfw.KEY_B and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
        
    if key == glfw.KEY_M and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    if restrictMode:
        cameraPos.x = max(CAM_MIN_X, min(CAM_MAX_X, cameraPos.x))
        cameraPos.y = max(CAM_MIN_Y, min(CAM_MAX_Y, cameraPos.y))
        cameraPos.z = max(CAM_MIN_Z, min(CAM_MAX_Z, cameraPos.z))

    # Manipulações de grupo exigidas pelo trabalho
    if restrictMode:
        
        # Aceleração [Onibus] (Translação) 
        if key == glfw.KEY_UP:
            if action == glfw.PRESS or action == glfw.REPEAT:
                onibus_acc = max(-onibus_max_acc, min(onibus_max_acc, onibus_acc - 0.001)) # Acelera pra frente
            elif action == glfw.RELEASE:
                onibus_acc = 0.0   # Para de acelerar quando solta a tecla
                
        if key == glfw.KEY_DOWN:
            if action == glfw.PRESS or action == glfw.REPEAT:
                onibus_acc = max(-onibus_max_acc, min(onibus_max_acc, onibus_acc + 0.001))  # Acelera pra trás
            elif action == glfw.RELEASE:
                onibus_acc = 0.0   # Para de acelerar quando solta a tecla

        # Aceleração [Spinner] (Rotação) 
        if key == glfw.KEY_LEFT:
            if action == glfw.PRESS or action == glfw.REPEAT:
                spinner_acc += -0.5 # Acelera horário
                # spinner_spd = 0.0
            elif action == glfw.RELEASE:
                spinner_spd = max(-spinner_max_speed, min(spinner_max_speed, spinner_acc)) # Aplica aceleração
                spinner_acc = 0.0   # Para de acelerar quando solta a tecla

        if key == glfw.KEY_RIGHT:
            if action == glfw.PRESS or action == glfw.REPEAT:
                spinner_acc += 0.5  # Acelera antihorário
                # spinner_spd = 0.0
            elif action == glfw.RELEASE:
                spinner_spd = max(-spinner_max_speed, min(spinner_max_speed, spinner_acc)) # Aplica aceleração
                spinner_acc = 0.0   # Para de acelerar quando solta a tecla

        # Escala (Relogio)
        if key == glfw.KEY_A and (action == glfw.PRESS): # liga/desliga alarme
            alarm_on = not alarm_on
        
    # Manipulações individuais utilizadas na montagem da cena
    else:
        # Muda Objeto
        if key == glfw.KEY_Y and action == glfw.PRESS:
            selected_obj = (selected_obj+1) % len(scene)

        # Toggle Visibility
        if key == glfw.KEY_V and action == glfw.PRESS: 
            scene[selected_obj]['visible'] = not scene[selected_obj]['visible']

        # Toggle uniform_scale
        if key == glfw.KEY_C and action == glfw.PRESS:
            uniform_scale = not uniform_scale

        # Troca de Eixo Rotação
        if key == glfw.KEY_Q and action == glfw.PRESS:
            eixo = (eixo+1)%3

        # Rotação
        if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['rotacao'][eixo] += 1.0
        if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['rotacao'][eixo] -= 1.0

        # Escala
        if uniform_scale:
            if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][0] += scene[selected_obj]['escala'][0] * 0.05
                scene[selected_obj]['escala'][1] += scene[selected_obj]['escala'][0] * 0.05
                scene[selected_obj]['escala'][2] += scene[selected_obj]['escala'][0] * 0.05
            if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][0] -= scene[selected_obj]['escala'][0] * 0.05
                scene[selected_obj]['escala'][1] -= scene[selected_obj]['escala'][0] * 0.05
                scene[selected_obj]['escala'][2] -= scene[selected_obj]['escala'][0] * 0.05
        else:
            if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][eixo] += 0.01
            if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['escala'][eixo] -= 0.01

        # Translação
        ## X
        if key == glfw.KEY_LEFT and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][0] -= 0.01
        if key == glfw.KEY_RIGHT and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][0] += 0.01
        ## Y
        if key == glfw.KEY_DOWN and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][1] -= 0.01
        if key == glfw.KEY_UP and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][1] += 0.01
        ## Z
        if key == glfw.KEY_Z and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][2] -= 0.01
        if key == glfw.KEY_X and (action == glfw.PRESS or action == glfw.REPEAT):
            scene[selected_obj]['translacao'][2] += 0.01
        

def framebuffer_size_callback(window, largura, altura):
    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0
    
glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Matrizes Model, View e Projection

In [250]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    # r_x, r_y, r_z são ângulos de Euler (em graus) em torno de X, Y, Z.
    # O parâmetro `angle` é mantido por compatibilidade com a assinatura,
    # mas não é mais usado (cada eixo tem seu próprio ângulo).

    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade

    # Ordem aplicada aos vértices: Scale -> Rotate -> Translate
    # (em GLM/coluna-major, basta multiplicar nesta ordem: T * R * S)

    # aplicando translacao
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))

    # aplicando rotacao em cada eixo (ângulos de Euler X, Y, Z)
    matrix_transform = glm.rotate(matrix_transform, math.radians(r_x), glm.vec3(1.0, 0.0, 0.0))
    matrix_transform = glm.rotate(matrix_transform, math.radians(r_y), glm.vec3(0.0, 1.0, 0.0))
    matrix_transform = glm.rotate(matrix_transform, math.radians(r_z), glm.vec3(0.0, 0.0, 1.0))

    # aplicando escala
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))

    matrix_transform = np.array(matrix_transform)

    return matrix_transform

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 100000.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

### Exibição e Loop Principal de Janela


In [251]:
glfw.show_window(window)

In [252]:
glEnable(GL_DEPTH_TEST)

# Atributos Onibus
onibus_acc = 0 # Aceleração do Onibus
onibus_spd = 0 # Velocidade do Onibus

onibus_max_acc = 0.005 # Aceleração máxima do onibus
onibus_max_speed = 0.5 # Velocidade máxima do onibus
onibus_atrito = 0.98

# Atributos Spinner
spinner_acc = 0 # Aceleração do Spinner
spinner_spd = 0 # Velocidade do Spinner

spinner_max_speed = 300.0 # Velocidade máxima do Spinner
spinner_atrito = 0.99

# Atributos Relógio
alarm_on = False

# Procura os objetos na cena
targets = ["Onibus", "spinner", "clock"]
targets_indexes = [None] * len(targets)

for i in range(len(targets)):
    for j, obj in enumerate(scene):
        if obj.get('name') == targets[i]:
            targets_indexes[i] = j
            break

# Loop principal, continua enquanto a janela estiver aberta
while not glfw.window_should_close(window):

    # Visibilidade do Onibus
    if targets_indexes[0] is not None:
        if scene[targets_indexes[0]]['translacao'][0] >= 270 or scene[targets_indexes[0]]['translacao'][0] <= -270:
            scene[targets_indexes[0]]['visible'] = False
        else:
            scene[targets_indexes[0]]['visible'] = True

    # Translação (Onibus)

    # Aumenta a velocidade de acordo com a aceleração
    # exceto quando alcança o limite de velocidade
    onibus_spd = max(-onibus_max_speed, min(onibus_max_speed, onibus_spd + onibus_acc))

    # Se aceleração = 0, começa a desacelar por atrito
    if onibus_acc == 0:
        # Zera speed (safeguard)
        if onibus_spd < 0.0005 and onibus_spd > -0.0005:
            onibus_spd = 0
        else:
            onibus_spd *= onibus_atrito

    # Zera speed (safeguard)
    if spinner_spd < 0.0005 and spinner_spd > -0.0005:
        spinner_spd = 0
    else:
        spinner_spd *= spinner_atrito * (0.98 if spinner_spd < 2.0 else 1)

    # Desloca o onibus de acordo com a velocidade
    if targets_indexes[0] is not None:
        scene[targets_indexes[0]]['translacao'][0] += onibus_spd
    # Roda o spinner de acordo com a velocidade
    if targets_indexes[1] is not None:
        scene[targets_indexes[1]]['rotacao'][1] += spinner_spd

    t = glfw.get_time()
    if alarm_on:
        sx = scene[targets_indexes[2]]['escala'][0] + 0.08 * math.sin(t * 20.0)
        sy = scene[targets_indexes[2]]['escala'][1] + 0.05 * math.sin(t * 25.0)
    else:
        sx = 1.0
        sy = 1.0
    scene[targets_indexes[2]]['escala'][0] = sx
    scene[targets_indexes[2]]['escala'][1] = sy
   
    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events() 
       
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(1.0, 1.0, 1.0, 1.0)

    # View e Projection calculados uma vez por frame
    mat_view = view()
    mat_projection = projection()

    # Shader
    mainShader.use() # Ativa o shader normal
    
    # Reconecta o buffer de vértices e texturas normais
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
    loc_vertices = glGetAttribLocation(program, "position")
    glEnableVertexAttribArray(loc_vertices)
    glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, vertices.strides[0], ctypes.c_void_p(0))

    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
    loc_texture_coord = glGetAttribLocation(program, "texture_coord")
    glEnableVertexAttribArray(loc_texture_coord)
    glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, textures.strides[0], ctypes.c_void_p(0))
    
    # Envia as matrizes para o shader normal
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

    # Para cada objeto na cena, realiza a lógica de renderização
    for i, object in enumerate(scene):

        # Controla o modo polígono
        if PolygonMode:
            glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
        else:
            glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

        # Ajusta nome da janela de acordo com o modo restrito
        if restrictMode:
            glfw.set_window_title(window, "Casa no Meio do Nada")
        else:
            glfw.set_window_title(window, f"name: {scene[selected_obj]['name']}, scene_id: {scene[selected_obj]['scene_id']}, uniform_scale: {uniform_scale}, eixo: {eixo}")

        # Pula a renderização do objeto se ele estiver marcado como 'não-visível'
        if not object['visible']:
            continue

        grupos  = verticeInicial_quantosVertices_list[i]
        angulo_obj = object["angulo_obj"]
        ax, ay, az = object['rotacao']
        tx, ty, tz = object['translacao']
        sx, sy, sz = object['escala']

        desenha_obj(angulo_obj, [ax, ay, az], [tx, ty, tz], [sx, sy, sz], grupos)
    
    # SkyBox
    glDepthFunc(GL_LEQUAL)  # Muda a função de profundidade
    skyboxShader.use()      # Ativa o shader do skybox
    
    # Remove a translação da matriz de View
    mat_view_skybox = glm.mat4(glm.mat3(view())) 
    mat_view_skybox = np.array(mat_view_skybox)
    
    # Envia as matrizes para o shader do skybox
    loc_view_sky = glGetUniformLocation(skyboxProgram, "view")
    glUniformMatrix4fv(loc_view_sky, 1, GL_TRUE, mat_view_skybox)
    loc_proj_sky = glGetUniformLocation(skyboxProgram, "projection")
    glUniformMatrix4fv(loc_proj_sky, 1, GL_TRUE, mat_projection)

    # Conecta o buffer exclusivo do Skybox
    glBindBuffer(GL_ARRAY_BUFFER, skybox_VBO)
    loc_sky_pos = glGetAttribLocation(skyboxProgram, "position")
    glEnableVertexAttribArray(loc_sky_pos)
    # Stride é 3 * 4 bytes (12 bytes por vértice, XYZ)
    glVertexAttribPointer(loc_sky_pos, 3, GL_FLOAT, False, 12, ctypes.c_void_p(0))
    
    # Ativa a textura Cubemap 
    glActiveTexture(GL_TEXTURE0)
    glBindTexture(GL_TEXTURE_CUBE_MAP, cubemap_texture_id)
    skyboxShader.setInt("imagem_cube", 0)

    # Desenha os 36 vértices do cubo
    glDrawArrays(GL_TRIANGLES, 0, 36)
    
    glDepthFunc(GL_LESS) # Retorna a função de profundidade ao normal para o próximo frame

    glfw.swap_buffers(window)


glfw.terminate()

if not restrictMode:
    save_scene(scene) # Salva a cena